## Notebook 5 - Parameterized Carbon Emissions Analytics

In this lab, we'll use Databricks Widgets to create a parameterized pipeline for CarbonOps.

The sustainability team wants to analyze carbon emissions for different cities and identify facilities exceeding a configurable emissions threshold.

Instead of modifying our PySpark code every time, we'll pass the city and minimum emissions threshold as parameters.

We'll use the existing carbonops.gold.facility_emissions table from the previous notebook.

### Create Widgets

In [ ]:
dbutils.widgets.text("city", "Mumbai")
dbutils.widgets.text("min_emissions", "100")

print("Widgets created successfully!")

### Load Gold Data

In [ ]:
from pyspark.sql.functions import (
    col, sum, avg, count, round as spark_round
)

df = spark.table("carbonops.gold.facility_emissions")

display(df)

### Calculate City-Level Carbon Emissions

In [ ]:
city_emissions_df = (
    df.groupBy("city")
    .agg(
        spark_round(sum("scope1_tco2e"), 2).alias("scope1_emissions"),
        spark_round(sum("scope2_tco2e"), 2).alias("scope2_emissions"),
        spark_round(sum("total_tco2e"), 2).alias("total_emissions")
    )
)

city_emissions_df.write.mode("overwrite") \
    .saveAsTable("carbonops.gold.city_emissions")

print("City emissions table created!")

### Calculate Facility Performance Metrics

In [ ]:
facility_perf_df = (
    df.groupBy("facility_id", "facility_name", "city")
    .agg(
        spark_round(sum("total_tco2e"), 2).alias("total_emissions"),
        spark_round(avg("total_tco2e"), 2).alias("avg_monthly_emissions"),
        count("*").alias("total_reporting_periods")
    )
)

facility_perf_df.write.mode("overwrite") \
    .saveAsTable("carbonops.gold.facility_performance")

print("Facility performance table created!")

### Calculate State-Level Carbon Emissions

In [ ]:
state_emissions_df = (
    df.groupBy("state")
    .agg(
        spark_round(sum("scope1_tco2e"), 2).alias("scope1_emissions"),
        spark_round(sum("scope2_tco2e"), 2).alias("scope2_emissions"),
        spark_round(sum("total_tco2e"), 2).alias("total_emissions")
    )
)

state_emissions_df.write.mode("overwrite") \
    .saveAsTable("carbonops.gold.state_emissions")

print("State emissions table created!")

### Read Parameters from Widgets

In [ ]:
city = dbutils.widgets.get("city")
min_emissions = float(dbutils.widgets.get("min_emissions"))

print("Selected City:", city)
print("Minimum Emissions Threshold:", min_emissions, "tCO2e")

### Create the Parameterized Emissions Table

In [ ]:
filtered_df = df.filter(
    (col("city") == city) &
    (col("total_tco2e") > min_emissions)
)

display(filtered_df)

### Write the Parameterized Emissions Table to Delta Lake

In [ ]:
import re

# Sanitize city for a valid table name
safe_city = re.sub(r"[^a-z0-9_]", "_", city.lower()).strip("_")

if not safe_city:
    raise ValueError("Please provide a valid city.")

parameterised_table_name = f"carbonops.gold.filtered_{safe_city}"

filtered_df.write.mode("overwrite") \
    .saveAsTable(parameterised_table_name)

print(f"Parameterized table created: {parameterised_table_name}")

### Validate Outputs

In [ ]:
display(spark.table("carbonops.gold.city_emissions"))

display(spark.table("carbonops.gold.facility_performance"))

display(spark.table("carbonops.gold.state_emissions"))

display(spark.table(parameterised_table_name))